# IndoML T2 — E13 (RunPod A100): train D to remove ONLY library events and keep Vaani-native audio (target = natural clip as-is) → TTA×4 test submission
Hypothesis: test synthetic = noisy natural Vaani clip + library events; reference = the natural clip (own noise kept). Leaders' ΔWER-syn ceiling (~17–18) = output ≈ reference.
Before running: upload `all_models_bundle.zip` to /workspace; paste tokens in cell 0. Download `e13_D.pt` + submission ZIPs before terminating.

In [ ]:
# 0) Setup on RunPod — paste your tokens below (do not share this notebook afterwards)
import os, sys, glob, json, subprocess, stat
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_KAGGLE_API_TOKEN_HERE'      # the KGAT_... token from Kaggle Settings -> API Tokens
assert os.environ['HF_TOKEN'].startswith('hf_'), 'paste your Hugging Face token'
assert os.environ['KAGGLE_API_TOKEN'].startswith('KGAT_'), 'paste your Kaggle API token'
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tok = os.path.expanduser('~/.kaggle/access_token'); open(tok, 'w').write(os.environ['KAGGLE_API_TOKEN']); os.chmod(tok, stat.S_IRUSR | stat.S_IWUSR)
for p_ in ['/workspace/work', '/workspace/input', '/kaggle']: os.makedirs(p_, exist_ok=True)
for src, dst in [('/workspace/work', '/kaggle/working'), ('/workspace/input', '/kaggle/input')]:
    if not os.path.exists(dst): os.symlink(src, dst)          # reuse the Kaggle code paths unchanged
subprocess.run('pip -q install -U kaggle jiwer sentencepiece webrtcvad-wheels librosa einops soundfile pandas scipy matplotlib joblib datasets transformers accelerate', shell=True)
print(subprocess.run('kaggle --version', shell=True, capture_output=True, text=True).stdout)
import torch; print(torch.__version__, torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9), 'GB')

In [ ]:
# 1) Data via the Kaggle API (avin55 token) + models from the bundle you upload to /workspace (avics55 datasets give 403)
from kaggle.api.kaggle_api_extended import KaggleApi
import zipfile
api = KaggleApi(); api.authenticate()
def get(refs, dest):
    os.makedirs(dest, exist_ok=True)
    for ref in refs:
        try: print('downloading', ref, flush=True); api.dataset_download_files(ref, path=dest, unzip=True, quiet=True); return
        except Exception as e: print('  failed', ref, str(e)[:100])
    raise RuntimeError(f'could not download {refs}')
get(['avin55/indoml-validation', 'avics55/indoml-validation'], '/workspace/input/datasets/avin55/indoml-validation')
get(['avin55/indoml-track2-test', 'avics55/indoml-track2-test'], '/workspace/input/datasets/avin55/indoml-track2-test')
get(['avics55/musan-noise', 'avin55/musan-noise'], '/workspace/input/musan-noise')
get(['avin55/indoml-t2-ckpt', 'avics55/indoml-t2-ckpt'], '/workspace/input/indoml-t2-ckpt')
assert os.path.exists('/workspace/all_models_bundle.zip'), 'upload all_models_bundle.zip to /workspace first'
os.makedirs('/workspace/input/mpsenet-e10r', exist_ok=True); zipfile.ZipFile('/workspace/all_models_bundle.zip').extractall('/workspace/input/mpsenet-e10r')
chk = {'validation': glob.glob('/workspace/input/**/validationMetadata.json', recursive=True),
       'test wavs (3178)': len(glob.glob('/workspace/input/datasets/avin55/indoml-track2-test/audio/*.wav')),
       'musan (930)': len(glob.glob('/workspace/input/musan-noise/**/*.wav', recursive=True)),
       'noisy asr': glob.glob('/workspace/input/**/noisy_asr_val.json', recursive=True),
       'models': sorted(os.path.basename(x) for x in glob.glob('/workspace/input/mpsenet-e10r/*.pt'))}
for k, v in chk.items(): print(f'{k:18s}:', v if v else 'MISSING')

In [ ]:
!rm -rf /kaggle/working/ner && git clone -q --depth 1 https://github.com/nagshubhadip/noise-event-detection-and-removal /kaggle/working/ner
!pip -q install jiwer sentencepiece webrtcvad-wheels librosa 2>&1 | tail -1
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /kaggle/working/esc50.zip && unzip -q -o /kaggle/working/esc50.zip -d /kaggle/working/ && ls /kaggle/working/ESC-50-master | head
# ---- MP-SENet (MIT) ----
!rm -rf /kaggle/working/mpsenet && git clone -q --depth 1 https://github.com/yxlu-0102/MP-SENet /kaggle/working/mpsenet
!pip -q install einops 2>&1 | tail -1

In [ ]:
import sys, os, glob, json, time, random, numpy as np, torch, soundfile as sf, librosa
from pathlib import Path
VAL = Path('/kaggle/input/datasets/avin55/indoml-validation/validation')
M = Path('/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular')
d = (M/'data.py').read_text()
d = d.replace('Path("/kaggle/input/indoml-validation"),', 'Path("/kaggle/input/datasets/avin55/indoml-validation/validation"), Path("/kaggle/input/indoml-validation"),')
d = d.replace('cid = (r.get("clip_id") or r.get("id")', 'cid = (r.get("segmentFileName") or r.get("clip_id") or r.get("id")')
(M/'data.py').write_text(d); sys.path.insert(0, str(M))
HF = os.environ["HF_TOKEN"]
import config, data, evaluate
import model as model_mod
from train import train_on_sim
from inference import enhance, Transcriber
from jiwer import wer as jwer
config.ensure_dirs()
val_dir, items = data.resolve_validation()
meta = {m['segmentFileName'][:-4]: m for m in json.load(open(VAL/'validationMetadata.json', encoding='utf-8'))}
print('validation clips', len(items))
import torch.nn.functional as F

In [ ]:
import types, sys
sys.modules.setdefault('pesq', types.SimpleNamespace(pesq=lambda *a, **k: 0.0))   # PESQ not needed (no metric discriminator)
sys.path.insert(0, '/kaggle/working/mpsenet')
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
h = AttrDict(json.load(open('/kaggle/working/mpsenet/best_ckpt/config.json')))
dev = torch.device('cuda')
def load_mp(path='/kaggle/working/mpsenet/best_ckpt/g_best_dns'):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)['generator']); return m.eval()
def mp_forward(m, wav):                       # wav [B, T] (RMS-normalised) -> enhanced [B, T]
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    amp_g, pha_g, com_g = m(amp, pha)
    return mag_pha_istft(amp_g, pha_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (amp_g, pha_g, com_g)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev)
    W, H = int(win * sr), int(hop * sr)
    if n <= W:
        y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); wsum = torch.zeros(n, device=dev)
        fade = torch.ones(W, device=dev); r = W - H; fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev)
        s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = torch.nn.functional.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]
            L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; wsum[s:e] += f
            if e == n: break
            s += H
        y = y / wsum.clamp_min(1e-6)
    y = (y / nf).cpu().numpy()
    return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
mp = load_mp(); print('MP-SENet params (M):', round(sum(p.numel() for p in mp.parameters()) / 1e6, 2))

In [ ]:
# Isolated validation noise events + fingerprint clustering
SR = 16000
NV = Path('/kaggle/working/bank/val_noise'); NV.mkdir(parents=True, exist_ok=True)
segs, fps, owner = [], [], []
for v in items:
    if not v['syn']: continue
    y = data.read_audio(v['path']); c = data.read_audio(v['ref']); L = min(len(y), len(c)); n = (y[:L] - c[:L]).astype(np.float32)
    for k, e in enumerate(meta.get(v['id'], {}).get('NoiseSubCategoryTimeStamp', [])):
        a, b = max(0, int((float(e['start']) - 0.02) * SR)), min(L, int((float(e['end']) + 0.02) * SR))
        if b - a < int(0.3 * SR) or np.abs(n[a:b]).max() < 1e-4: continue
        seg = n[a:b]; p = NV / f"{v['id'][-40:]}_{k}.wav"; sf.write(p, seg, SR)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=48, n_fft=512, hop_length=160) + 1e-10)
        f = librosa.util.fix_length(mel, size=48, axis=1).ravel(); f = (f - f.mean()) / (f.std() + 1e-6)
        segs.append({'path': str(p), 'dur': (b - a) / SR}); fps.append(f / np.linalg.norm(f)); owner.append(v['id'])
F = np.stack(fps); cents, lab = [], np.zeros(len(F), int)
for i, f in enumerate(F):
    if cents:
        s = np.stack(cents) @ f; j = int(s.argmax())
        if s[j] > 0.95: lab[i] = j; continue
    cents.append(f); lab[i] = len(cents) - 1
ncl = len(cents); rng = np.random.default_rng(7)
print('noise events', len(F), '| clusters', ncl)
for frac in [0.35, 0.45, 0.55]:
    held = set(rng.permutation(ncl)[:int(frac * ncl)].tolist())
    clip_cl = {}
    for o, l in zip(owner, lab): clip_cl.setdefault(o, set()).add(int(l))
    HELD_CLIPS = [o for o, s in clip_cl.items() if s <= held]
    if len(HELD_CLIPS) >= 80: break
train_noise = [s for s, l in zip(segs, lab) if int(l) not in held]
print(f'held-out clusters {len(held)}/{ncl} | held-out eval clips {len(HELD_CLIPS)} | train noise segments {len(train_noise)}')

In [ ]:
# ESC-50 noise segments (active part, 0.3-2.5 s), resampled to 16 kHz
E = Path('/kaggle/working/ESC-50-master'); NE = Path('/kaggle/working/bank/esc50'); NE.mkdir(parents=True, exist_ok=True)
import pandas as pd
esc = pd.read_csv(E/'meta/esc50.csv'); esc_noise = []
for _, r in esc.iterrows():
    w, _ = librosa.load(E/'audio'/r.filename, sr=SR, mono=True)
    w, _ = librosa.effects.trim(w, top_db=30)
    if len(w) < int(0.3 * SR): continue
    w = w[:int(2.5 * SR)].astype(np.float32); p = NE / r.filename; sf.write(p, w, SR)
    esc_noise.append({'path': str(p), 'dur': len(w) / SR, 'cat': r.category})
print('ESC-50 segments', len(esc_noise), '| categories', esc.category.nunique())

In [ ]:
# MUSAN noise from the uploaded Kaggle dataset (no download)
MUF = sorted(glob.glob('/kaggle/input/**/free-sound/*.wav', recursive=True) + glob.glob('/kaggle/input/**/sound-bible/*.wav', recursive=True))
print('MUSAN wav files found:', len(MUF), '(expect 930)'); assert len(MUF) > 800, 'attach the musan-noise dataset'
NM = Path('/kaggle/working/bank/musan'); NM.mkdir(parents=True, exist_ok=True); rngm = np.random.default_rng(3); musan_noise = []
for k, fpath in enumerate(MUF):
    w, _ = librosa.load(fpath, sr=SR, mono=True)
    if len(w) < int(0.5 * SR) or np.abs(w).max() < 1e-4: continue
    for j in range(min(4, max(1, len(w) // (3 * SR)))):
        L = int(rngm.uniform(0.5, 2.5) * SR); st = int(rngm.integers(0, max(1, len(w) - L)))
        seg = w[st:st + L].astype(np.float32)
        if np.sqrt((seg**2).mean()) < 1e-3: continue
        p = NM / f'{k:04d}_{j}.wav'; sf.write(p, seg, SR); musan_noise.append({'path': str(p), 'dur': len(seg) / SR})
print('MUSAN noise segments:', len(musan_noise))

In [ ]:
# Noise augmentation: speed perturbation, random EQ, synthetic reverb -> one augmented copy per base segment
from scipy.signal import butter, sosfilt, fftconvolve
NA = Path('/kaggle/working/bank/aug'); NA.mkdir(parents=True, exist_ok=True); rnga = np.random.default_rng(11)
def augment(w):
    f = rnga.choice([0.9, 1.0, 1.1]);  w = librosa.resample(w, orig_sr=int(SR * f), target_sr=SR) if f != 1.0 else w
    if rnga.random() < 0.6:
        if rnga.random() < 0.5: sos = butter(4, rnga.uniform(2000, 7000), 'low', fs=SR, output='sos')
        else: sos = butter(2, rnga.uniform(100, 500), 'high', fs=SR, output='sos')
        w = sosfilt(sos, w)
    if rnga.random() < 0.5:
        rt = rnga.uniform(0.15, 0.7); t = np.arange(int(rt * SR)) / SR
        rir = rnga.standard_normal(len(t)) * np.exp(-6.9 * t / rt); rir[0] = 1.0
        w = fftconvolve(w, rir / np.abs(rir).sum() * 4)[:len(w)]
    pk = np.abs(w).max(); return (w / pk * 0.9 if pk > 0 else w).astype(np.float32)
base = train_noise + esc_noise + musan_noise; aug_noise = []
for k, nb_ in enumerate(base):
    w, _ = sf.read(nb_['path'], dtype='float32'); w = augment(w)
    if len(w) < int(0.3 * SR): continue
    p = NA / f'a{k:05d}.wav'; sf.write(p, w, SR); aug_noise.append({'path': str(p), 'dur': len(w) / SR})
print('noise pool:', len(base), 'base +', len(aug_noise), 'augmented')

In [ ]:
# Load model D (E10-R annealed) from the uploaded bundle dataset; rebuild the .pt if Kaggle extracted it
import zipfile, gc, torch.nn.functional as F
def find_ckpt(key):
    plain = [x for x in glob.glob(f'/kaggle/input/**/*{key}*.pt', recursive=True) if os.path.isfile(x)]
    if plain: return plain[0]
    roots = [Path(x).parent for x in glob.glob('/kaggle/input/**/data.pkl', recursive=True) if key in x]
    assert roots, f'{key} not found: attach the mpsenet-e10r dataset (all_models_bundle.zip)'
    out = Path(f'/kaggle/working/{key}_rebuilt.pt')
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:
        for f in sorted(roots[0].rglob('*')):
            if f.is_file(): z.write(f, f'{roots[0].name}/{f.relative_to(roots[0]).as_posix()}')
    return str(out)
D_CK = find_ckpt('anneal'); B_CK = find_ckpt('warm_e8')
mD = load_mp(D_CK); print('D:', D_CK, '| B:', B_CK)
del mp; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 2) CHECK: how much does D alter Vaani-native audio? SI-SDR(reference, D(reference)) on validation refs and natural clips (higher = D leaves it alone)
byid = {v['id']: v for v in items}
def alter(p): r = data.read_audio(p); return float(evaluate.si_sdr(r, enhance_mp(mD, r, win=8.0, hop=6.0)))
refs = [v['ref'] for v in items if v['syn']][:200]; nats = [v['path'] for v in items if not v['syn']][:300]
a_ref = np.array([alter(p) for p in refs]); a_nat = np.array([alter(p) for p in nats])
print(f'D on VAL synthetic REFERENCES: SI-SDR(ref, D(ref)) p25 {np.percentile(a_ref,25):.1f} | p50 {np.percentile(a_ref,50):.1f} | p75 {np.percentile(a_ref,75):.1f} dB')
print(f'D on VAL natural clips       : SI-SDR(x, D(x))     p25 {np.percentile(a_nat,25):.1f} | p50 {np.percentile(a_nat,50):.1f} | p75 {np.percentile(a_nat,75):.1f} dB')
print('If test references are NOISY natural clips, a model that strips their native noise is penalised; E13 trains D to remove only library events.')

In [ ]:
# 3) E13 data: TARGET = natural Vaani clip as-is (own ambience/noise kept); INPUT = target + 1-3 LIBRARY events (ESC-50 / MUSAN / validation library + augmented)
import gc
from datasets import load_dataset, Audio
rng = np.random.RandomState(0); NAT_ALL = [v['id'] for v in items if not v['syn']]
NS_EVAL = set(NAT_ALL[i] for i in rng.choice(len(NAT_ALL), 600, replace=False))       # 600 natural clips reserved for evaluation
targets = [data.read_audio(byid[i]['path']) for i in NAT_ALL if i not in NS_EVAL]
print('validation natural targets', len(targets), flush=True)
VAANI_EVAL = []
raw = load_dataset(config.REPO, split='train', streaming=True, token=HF).cast_column('audio', Audio(decode=False)); n = 0; t0 = time.time()
for ex in raw:
    try: w = data.decode_audio(ex['audio'])
    except Exception: continue
    if w is None or len(w) < 2 * SR: continue
    if len(VAANI_EVAL) < 400: VAANI_EVAL.append(w[:int(12 * SR)].astype(np.float32)); continue     # reserved NOISY Vaani clips for the test-like eval
    targets.append(w[:int(12 * SR)].astype(np.float32)); n += 1
    if n >= 9000: break
print(f'+ {n} Vaani training clips as targets in {(time.time()-t0)/60:.1f} min | total targets {len(targets)}', flush=True)
import pandas as pd
escm = pd.read_csv('/kaggle/working/ESC-50-master/meta/esc50.csv').set_index('filename')
def esc_fold(p): f = os.path.basename(p); return int(escm.loc[f]['fold']) if f in escm.index else 0
excl = set(len(train_noise) + j for j, e in enumerate(esc_noise) if esc_fold(e['path']) == 5) | set(range(len(train_noise) + len(esc_noise) + 2100, len(train_noise) + len(esc_noise) + len(musan_noise)))
aug_ok = [a for a in aug_noise if int(os.path.basename(a['path'])[1:6]) not in excl]      # drop augmented copies of the eval events (no leakage)
ev_train = [e for e in esc_noise if esc_fold(e['path']) != 5] + musan_noise[:2100] + train_noise + aug_ok
ev_eval = [e for e in esc_noise if esc_fold(e['path']) == 5] + musan_noise[2100:]
print('event pool train', len(ev_train), '| event pool eval (unseen)', len(ev_eval))
RNG = np.random.default_rng(42)
def rd(p): w, _ = sf.read(p, dtype='float32'); return w if w.ndim == 1 else w.mean(1)
def make(pool, n_ev_choices=(1, 1, 2, 2, 3)):
    s = targets[RNG.integers(len(targets))]
    if len(s) < 6 * SR: s = np.concatenate([s, targets[RNG.integers(len(targets))]])
    s = s[:int(12 * SR)].astype(np.float32); L = len(s); ps = float((s**2).mean()) + 1e-10; x = s.copy(); ev = []
    for _ in range(int(RNG.choice(n_ev_choices))):
        e = rd(pool[RNG.integers(len(pool))]['path'])[:int(2.5 * SR)]
        if len(e) < int(0.3 * SR) or len(e) >= L: continue
        on = int(RNG.integers(0, L - len(e))); snr = RNG.uniform(-20, 5)
        x[on:on + len(e)] += e * np.sqrt(ps / ((e**2).mean() + 1e-10) / 10**(snr / 10)); ev.append({'onset': on / SR, 'offset': (on + len(e)) / SR})
    pk = max(np.abs(x).max(), 1e-6); g = 0.95 / pk if pk > 0.95 else 1.0
    return (x * g).astype(np.float32), (s * g).astype(np.float32), ev
for f in list(config.MIX_DIR.glob('*.wav')) + list(config.REF_DIR.glob('*.wav')): f.unlink()
records = []
for i in range(30000):
    x, s, ev = make(ev_train)
    if not ev: continue
    cid = f'e13_{i:06d}'; sf.write(config.MIX_DIR / f'{cid}.wav', x, SR); sf.write(config.REF_DIR / f'{cid}.wav', s, SR)
    records.append({'id': cid, 'mix': str(config.MIX_DIR / f'{cid}.wav'), 'clean': str(config.REF_DIR / f'{cid}.wav'), 'events': ev})
    if i % 5000 == 0: print(i, flush=True)
with open(config.MANIFEST, 'w') as f:
    for rr in records: f.write(json.dumps(rr) + '\n')
print('E13 mixtures', len(records), '| input SI-SDR', round(float(np.mean([evaluate.si_sdr(rd(rr['clean']), rd(rr['mix'])) for rr in records[:300]])), 2), 'dB')
json.dump(HELD_CLIPS, open('/kaggle/working/held_clips.json', 'w')); del targets; gc.collect()

In [ ]:
SCRIPT = r'''
import sys, os, json, time, glob, argparse, types, numpy as np, torch, torch.nn.functional as F
ap = argparse.ArgumentParser(); ap.add_argument("--lam", type=float); ap.add_argument("--epochs", type=int, default=5)
ap.add_argument("--out"); ap.add_argument("--bs", type=int, default=2); ap.add_argument("--seg", type=int, default=16000)
ap.add_argument("--lr", type=float, default=2e-4); ap.add_argument("--hours", type=float, default=9.0)
ap.add_argument("--init", default="/kaggle/working/mpsenet/best_ckpt/g_best_dns"); ap.add_argument("--bf16", type=int, default=0); ap.add_argument("--max_steps", type=int, default=0); a = ap.parse_args()
M = "/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular"; sys.path.insert(0, M)
sys.modules.setdefault("pesq", types.SimpleNamespace(pesq=lambda *x, **k: 0.0)); sys.path.append("/kaggle/working/mpsenet")   # append: MP-SENet also has an inference.py
HF = os.environ["HF_TOKEN"]
import config, data, evaluate
import model as model_mod
from inference import Transcriber
from jiwer import wer as jwer
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
torch.manual_seed(0); np.random.seed(0)
h = AttrDict(json.load(open("/kaggle/working/mpsenet/best_ckpt/config.json"))); dev = torch.device("cuda")
def load_mp(path="/kaggle/working/mpsenet/best_ckpt/g_best_dns"):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)["generator"]); return m
def mp_forward(m, wav):
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    a, p, c = m(amp, pha); return mag_pha_istft(a, p, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (a, p, c)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev); W, H = int(win*sr), int(hop*sr)
    if n <= W: y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); ws = torch.zeros(n, device=dev); r = W - H
        fade = torch.ones(W, device=dev); fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev); s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = F.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]; L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; ws[s:e] += f
            if e == n: break
            s += H
        y = y / ws.clamp_min(1e-6)
    y = (y / nf).cpu().numpy(); return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
def post(x, y):
    ri, ro = np.sqrt((x**2).mean()), np.sqrt((y**2).mean()); z = y * (ri / ro) if ro > 1e-8 else y
    pk = np.abs(z).max(); return (z * (0.99 / pk) if pk > 0.99 else z).astype(np.float32)
data.N_SAMP = int(os.environ.get("NSAMP", data.N_SAMP))
records = [json.loads(l) for l in open(config.MANIFEST)]
tdl, _, _, _ = data.build_dataloaders(records, a.bs)
encoder = enc_fwd = None
if a.lam > 0: _, encoder, _ = model_mod.core_model(HF); enc_fwd = type(encoder).forward.__wrapped__
val_dir, items = data.resolve_validation(); byid = {v["id"]: v for v in items}
H = json.load(open("/kaggle/working/held_clips.json")); X = {i: data.read_audio(byid[i]["path"]) for i in H}; REF = {i: data.read_audio(byid[i]["ref"]) for i in H}
def heldout_sisdr(m):
    m.eval(); s = float(np.mean([evaluate.si_sdr(REF[i], post(X[i], enhance_mp(m, X[i]))) for i in H])); m.train(); return s
m = load_mp(a.init); m.train(); opt = torch.optim.AdamW(m.parameters(), lr=a.lr, betas=(0.8, 0.99))
total = a.epochs * len(tdl)
if a.max_steps: total = min(total, a.max_steps)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=total, pct_start=0.05)
best = heldout_sisdr(m); print(f"[lam={a.lam}] init ({os.path.basename(a.init)}) held-out SI-SDR {best:.2f}", flush=True); torch.save({"generator": m.state_dict()}, a.out)
t0 = time.time(); step = 0; stop = False
for ep in range(a.epochs):
    run = None
    for b in tdl:
        mix, clean = b["mix"].to(dev), b["clean"].to(dev)
        if mix.shape[-1] > a.seg: st = np.random.randint(0, mix.shape[-1] - a.seg); mix, clean = mix[:, st:st+a.seg], clean[:, st:st+a.seg]
        nf = torch.sqrt(mix.shape[-1] / (mix.pow(2).sum(-1, keepdim=True) + 1e-9)); mix, clean = mix * nf, clean * nf
        c_amp, c_pha, c_com = mag_pha_stft(clean, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        n_amp, n_pha, _ = mag_pha_stft(mix, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=bool(a.bf16)):
            a_g, p_g, co_g = m(n_amp, n_pha)
        a_g, p_g, co_g = a_g.float(), p_g.float(), co_g.float()
        y = mag_pha_istft(a_g, p_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor)[:, :clean.shape[-1]]
        _, _, co_h = mag_pha_stft(y, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        ip, gd, iaf = phase_losses(c_pha, p_g)
        loss = 0.9*F.mse_loss(c_amp, a_g) + 0.3*(ip+gd+iaf) + 0.2*F.mse_loss(c_com, co_g) + 0.2*F.mse_loss(co_g, co_h) + 0.2*F.l1_loss(clean, y)
        if a.lam > 0:
            with torch.no_grad(): fc = enc_fwd(encoder, clean / nf); fc = F.layer_norm(fc.float(), fc.shape[-1:])
            fe = enc_fwd(encoder, y / nf); fe = F.layer_norm(fe.float(), fe.shape[-1:]); loss = loss + a.lam * (fe - fc).abs().mean()
        opt.zero_grad(set_to_none=True); loss.backward(); gn = torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
        if torch.isfinite(gn): opt.step()
        sched.step(); step += 1; run = loss.item() if run is None else 0.99 * run + 0.01 * loss.item()
        if step % 500 == 0: print(f"[lam={a.lam}] ep{ep} step{step}/{total} loss {run:.4f} | {(time.time()-t0)/3600:.2f} h", flush=True)
        if time.time() - t0 > a.hours * 3600 or step >= total: stop = True; break
    s = heldout_sisdr(m); print(f"[lam={a.lam}] epoch {ep} held-out SI-SDR {s:.2f} (best {best:.2f})", flush=True)
    if s > best: best = s
    torch.save({"generator": m.state_dict()}, a.out)   # keep LAST (annealed; held-out is val-like, not test-like)
    if stop: print("time limit reached", flush=True); break
# final leak-free eval of the best checkpoint: held-out synthetic + ALL natural
del encoder; torch.cuda.empty_cache()
m = load_mp(a.out).eval(); noisy = json.load(open(os.environ["NOISY_ASR"])); asr = Transcriber(HF)
NAT = [v["id"] for v in items if not v["syn"]]; GT = {i: evaluate.normalize(byid[i]["gt"]) for i in H + NAT}
for i in NAT: X[i] = data.read_audio(byid[i]["path"])
def pooled(ids, hyp): ids = [i for i in ids if GT[i]]; return jwer([GT[i] for i in ids], [evaluate.normalize(hyp.get(i, "")) or "@" for i in ids])
ENH = {i: enhance_mp(m, X[i]) for i in H + NAT}
for bl in [1.0]:
    out = {i: post(X[i], bl * ENH[i] + (1 - bl) * X[i]) for i in H + NAT}
    si = float(np.mean([evaluate.si_sdr(REF[i], out[i]) for i in H])); hyp = {}; ids = H + NAT
    for k in range(0, len(ids), 16):
        for i, t in zip(ids[k:k+16], asr.transcribe_many([out[i] for i in ids[k:k+16]])): hyp[i] = t
    ds, dn = 100 * (pooled(H, noisy) - pooled(H, hyp)), 100 * (pooled(NAT, noisy) - pooled(NAT, hyp))
    print("[RESULT]", json.dumps(dict(run=os.path.basename(a.out), lam=a.lam, blend=bl, si_sdr_heldout=round(si, 2), dWER_syn_heldout=round(ds, 2), dWER_nat_all=round(dn, 2), combined_est=round(si + 0.32*ds + 0.68*dn, 2))), flush=True)
'''

In [ ]:
open('/kaggle/working/mp_ft.py', 'w').write(SCRIPT); gc.collect(); torch.cuda.empty_cache(); print('script written')

In [ ]:
# 4) Memory + SPEED probe -> choose batch/segment and SIZE THE SCHEDULE to the real throughput (lesson from E10-R)
mp_probe = load_mp(D_CK); mp_probe.train(); opt_p = torch.optim.AdamW(mp_probe.parameters(), lr=1e-6); CFG = None
for bs, seg in [(2, 96000), (3, 64000), (4, 48000)]:
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); ts = []
        for it in range(6):
            t1 = time.time(); mix = torch.randn(bs, seg, device=dev) * 0.1
            amp, pha, _ = mag_pha_stft(mix, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
            with torch.autocast('cuda', dtype=torch.bfloat16): a_g, p_g, c_g = mp_probe(amp, pha)
            (a_g.float().mean() + p_g.float().mean() + c_g.float().mean()).backward(); opt_p.step(); opt_p.zero_grad(set_to_none=True); torch.cuda.synchronize(); ts.append(time.time() - t1)
        peak = torch.cuda.max_memory_allocated() / 1e9; sph = 3600 / np.mean(ts[2:]) * 0.80     # 20% margin for data loading + losses
        print(f'bs {bs} x {seg/16000:.0f}s -> peak {peak:.1f} GB | ~{sph:.0f} steps/h', flush=True)
        if peak < 0.8 * torch.cuda.get_device_properties(0).total_memory / 1e9: CFG = (bs, seg, sph); break
    except torch.cuda.OutOfMemoryError: print(f'bs {bs} x {seg/16000:.0f}s -> OOM', flush=True)
    finally: gc.collect(); torch.cuda.empty_cache()
del mp_probe, opt_p; gc.collect(); torch.cuda.empty_cache()
BUDGET_H = 2.75; MAX_STEPS = int(BUDGET_H * CFG[2]); print('chosen', CFG[:2], '| MAX_STEPS', MAX_STEPS, f'for {BUDGET_H} h')

In [ ]:
# 5) Fine-tune D on the TEST-LIKE mixtures (LR 7e-5 OneCycle over exactly MAX_STEPS -> anneals to ~0), bf16, last checkpoint kept
NOISY = glob.glob('/kaggle/input/**/noisy_asr_val.json', recursive=True)[0]
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', NOISY_ASR=NOISY, NSAMP=str(160000), PYTORCH_ALLOC_CONF='expandable_segments:True')
cmd = ['python', '/kaggle/working/mp_ft.py', '--lam', '0', '--out', '/workspace/work/e13_D.pt', '--init', D_CK, '--bf16', '1',
       '--bs', str(CFG[0]), '--seg', str(CFG[1]), '--epochs', '20', '--max_steps', str(MAX_STEPS), '--lr', '7e-5', '--hours', str(BUDGET_H + 0.4)]
print(' '.join(cmd)); p = subprocess.Popen(cmd, stdout=open('/workspace/work/e13.log', 'w'), stderr=subprocess.STDOUT, env=env)
while p.poll() is None:
    time.sleep(600); L = [l for l in open('/workspace/work/e13.log').read().splitlines() if l.startswith('[lam')]
    print(time.strftime('%H:%M'), L[-1] if L else '', flush=True)
print('exit', p.returncode); print('\n'.join(l for l in open('/workspace/work/e13.log').read().splitlines() if 'held-out' in l or l.startswith(('[RESULT]', 'Traceback'))))
print('>>> e13_D.pt saved in /workspace/work — download it <<<')

In [ ]:
# 6) TEST-LIKE evaluation: the 600 reserved natural clips + UNSEEN library events (ESC fold 5 + held MUSAN); target = the natural clip. D vs E13, win8 + TTA2.
mE = load_mp('/workspace/work/e13_D.pt'); from inference import Transcriber; asr = Transcriber(HF); noisy = json.load(open(glob.glob('/kaggle/input/**/noisy_asr_val.json', recursive=True)[0]))
NS = sorted(NS_EVAL); RNG = np.random.default_rng(9); LX, LT = {}, {}
for i in NS:
    s = data.read_audio(byid[i]['path']); L = len(s); ps = float((s**2).mean()) + 1e-10; x = s.copy()
    for _ in range(int(RNG.choice([1, 1, 2]))):
        e = rd(ev_eval[RNG.integers(len(ev_eval))]['path'])[:int(2.5 * SR)]
        if len(e) < int(0.3 * SR) or len(e) >= L: continue
        on = int(RNG.integers(0, L - len(e))); x[on:on + len(e)] += e * np.sqrt(ps / ((e**2).mean() + 1e-10) / 10**(RNG.uniform(-20, 5) / 10))
    LX[i], LT[i] = x.astype(np.float32), s.astype(np.float32)
GT = {i: evaluate.normalize(byid[i]['gt']) for i in NS}
def pooled(hyp): k_ = [i for i in NS if GT[i]]; return jwer([GT[i] for i in k_], [evaluate.normalize(hyp.get(i, '')) or '@' for i in k_])
def post(x, y):
    ri, ro = np.sqrt((x**2).mean()), np.sqrt((y**2).mean()); z = y * (ri / ro) if ro > 1e-8 else y
    pk = np.abs(z).max(); return (z * (0.99 / pk) if pk > 0.99 else z).astype(np.float32)
def tta(mm, x, shifts=(0, 50)): return np.mean([enhance_mp(mm, np.pad(x, (s_, 0)), win=8.0, hop=6.0)[s_:s_ + len(x)] for s_ in shifts], 0).astype(np.float32)
def asr_all(sig):
    hyp = {}
    for j in range(0, len(NS), 16):
        for i, t in zip(NS[j:j+16], asr.transcribe_many([sig[i] for i in NS[j:j+16]])): hyp[i] = t
    return hyp
W_IN = pooled(asr_all(LX)); print('input (natural + library events): WER', round(W_IN, 4), '| input SI-SDR', round(float(np.mean([evaluate.si_sdr(LT[i], LX[i]) for i in NS])), 2))
H = [i for i in HELD_CLIPS if i in byid]
for name, mm in [('D (current)', mD), ('E13', mE)]:
    out = {i: post(LX[i], tta(mm, LX[i])) for i in NS}
    si = float(np.mean([evaluate.si_sdr(LT[i], out[i]) for i in NS])); dw = 100 * (W_IN - pooled(asr_all(out)))
    si_h = float(np.mean([evaluate.si_sdr(data.read_audio(byid[i]['ref']), post(data.read_audio(byid[i]['path']), tta(mm, data.read_audio(byid[i]['path'])))) for i in H]))
    print(f'{name:12s} TEST-LIKE (natural+events) SI-SDR {si:.2f} | dWER {dw:+.2f} | val held-out SI-SDR {si_h:.2f}', flush=True)
# --- STRESS TEST 1: 12-s test-like clips (3 natural clips concatenated + events): window 8 s vs full-clip (16 s), TTA2, for D and E13
RNG = np.random.default_rng(21); LLX, LLT = {}, {}
for k in range(150):
    ids3 = list(RNG.choice(NS, 3, replace=False)); s = np.concatenate([data.read_audio(byid[i]['path']) for i in ids3]); L = len(s); ps = float((s**2).mean()) + 1e-10; x = s.copy()
    for _ in range(int(RNG.choice([1, 2, 2, 3]))):
        e = rd(ev_eval[RNG.integers(len(ev_eval))]['path'])[:int(2.5 * SR)]
        if len(e) < int(0.3 * SR) or len(e) >= L: continue
        on = int(RNG.integers(0, L - len(e))); x[on:on + len(e)] += e * np.sqrt(ps / ((e**2).mean() + 1e-10) / 10**(RNG.uniform(-20, 5) / 10))
    LLX[k], LLT[k] = x.astype(np.float32), s.astype(np.float32)
print('long test-like clips: median', round(float(np.median([len(LLX[k]) / SR for k in LLX])), 1), 's')
WINS = {}
for name, mm in [('D', mD), ('E13', mE)]:
    for win, hop in [(8.0, 6.0), (16.0, 12.0)]:
        si = float(np.mean([evaluate.si_sdr(LLT[k], post(LLX[k], np.mean([enhance_mp(mm, np.pad(LLX[k], (s_, 0)), win=win, hop=hop)[s_:s_ + len(LLX[k])] for s_ in (0, 50)], 0))) for k in LLX]))
        WINS[(name, win)] = si; print(f'{name:4s} win{int(win):2d} + TTA2 on 12-s clips: SI-SDR {si:.2f}', flush=True)
# --- STRESS TEST 2: mandated ASR batched (16) vs one-by-one on 300 natural clips (padding effects)
sub_ids = NS[:300]; sig = {i: data.read_audio(byid[i]['path']) for i in sub_ids}
hb = {}
for j in range(0, len(sub_ids), 16):
    for i, t in zip(sub_ids[j:j+16], asr.transcribe_many([sig[i] for i in sub_ids[j:j+16]])): hb[i] = t
hs = {i: asr.transcribe_many([sig[i]])[0] for i in sub_ids}
k_ = [i for i in sub_ids if GT[i]]
wb = jwer([GT[i] for i in k_], [evaluate.normalize(hb.get(i, '')) or '@' for i in k_]); ws = jwer([GT[i] for i in k_], [evaluate.normalize(hs.get(i, '')) or '@' for i in k_])
ASR_BS = 1 if ws < wb - 0.0005 else 16
print(f'ASR WER batched(16) {wb:.4f} vs one-by-one {ws:.4f} -> test transcription batch size {ASR_BS}')
# --- KEY CHECK: NOISY Vaani clips (reserved, unseen) + unseen library events, target = the Vaani clip as-is. If D lands near the LB (~13 dB) here, this set mimics the test.
RNG = np.random.default_rng(33); VX, VT = [], []
for s in VAANI_EVAL:
    L = len(s); ps = float((s**2).mean()) + 1e-10; x = s.copy()
    for _ in range(int(RNG.choice([1, 2, 2, 3]))):
        ev = rd(ev_eval[RNG.integers(len(ev_eval))]['path'])[:int(2.5 * SR)]
        if len(ev) < int(0.3 * SR) or len(ev) >= L: continue
        on = int(RNG.integers(0, L - len(ev))); x[on:on + len(ev)] += ev * np.sqrt(ps / ((ev**2).mean() + 1e-10) / 10**(RNG.uniform(-20, 5) / 10))
    VX.append(x.astype(np.float32)); VT.append(s)
print('noisy-Vaani test-like set:', len(VX), 'clips | input SI-SDR', round(float(np.mean([evaluate.si_sdr(t, x) for t, x in zip(VT, VX)])), 2))
VSI = {}
for name, mm in [('D', mD), ('E13', mE)]:
    VSI[name] = float(np.mean([evaluate.si_sdr(t, post(x, tta(mm, x))) for t, x in zip(VT, VX)]))
    print(f'{name:4s} on NOISY-Vaani test-like set: SI-SDR {VSI[name]:.2f}   (LB SI-SDR of D-type models: 13.18)', flush=True)
print('E13 gain on the set that mimics the test:', round(VSI['E13'] - VSI['D'], 2), 'dB')

In [ ]:
# 7) Choose the model for the test run: E13 if it wins on the TEST-LIKE set (that set mirrors the diagnosed test structure)
USE_E13 = True        # keep True only if E13 beats D on the NOISY-Vaani line (the set that mimics the test)        # set False if E12 does not beat D on the TEST-LIKE line above
MODELS = {'M': mE if USE_E13 else mD}; BEST_PARTS = ['M']; BEST_WIN = max([8.0, 16.0], key=lambda w: WINS[('E13' if USE_E13 else 'D', w)]); BEST_KW = dict(shifts=(0, 25, 50, 75), win=BEST_WIN, hop=BEST_WIN * 0.75)
def enhance_tta(mm, x, shifts=(0,), win=4.0, hop=3.0):
    return np.mean([enhance_mp(mm, np.pad(x, (s_, 0)), win=win, hop=hop)[s_:s_ + len(x)] for s_ in shifts], 0).astype(np.float32)
from inference import check_output, Transcriber
asr = Transcriber(HF); print('test model:', 'E13' if USE_E13 else 'D', '| TTA x4, win', BEST_WIN, '| ASR batch', ASR_BS)

In [ ]:
import soundfile as sf, zipfile, time
from collections import Counter
from pathlib import Path
test_files = sorted(Path('/kaggle/input/datasets/avin55/indoml-track2-test/audio').glob('*.wav')); print('test clips', len(test_files))
sub = Path('/workspace/work/t2_submission'); sub.mkdir(exist_ok=True); fails = Counter(); t0 = time.time(); parts = BEST_PARTS
for k, p in enumerate(test_files):
    w = data.read_audio(p); y = post(w, np.mean([enhance_tta(MODELS[q], w, **BEST_KW) for q in parts], 0)); c = check_output(w, y)
    for key in ['finite', 'len_match', 'mono', 'not_silent', 'level_ok', 'no_clip']:
        if not c[key]: fails[key] += 1
    assert c['finite'] and c['len_match'] and c['mono'], p.name
    sf.write(sub / p.name, y, 16000, subtype='PCM_16')
    if k % 500 == 0: print(k, f'{(time.time()-t0)/60:.1f} min', flush=True)
print('enhanced in', round((time.time()-t0)/60, 1), 'min | fails', dict(fails), '| combo', BEST)
names = [p.name for p in test_files]; texts = {}
for i in range(0, len(names), ASR_BS):
    for n, t in zip(names[i:i+ASR_BS], asr.transcribe_paths([str(sub / n) for n in names[i:i+ASR_BS]])): texts[n[:-4]] = (t or '').strip()
with open('/workspace/work/transcripts.jsonl', 'w', encoding='utf-8') as f:
    for n in names: f.write(json.dumps({'clip_id': n[:-4], 'text': texts.get(n[:-4], '')}, ensure_ascii=False) + '\n')
zp = '/workspace/work/submission_e13.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/workspace/work/transcripts.jsonl', 'transcripts.jsonl')
    for n in names: z.write(sub / n, n)
nl = zipfile.ZipFile(zp).namelist(); print('ZIP entries', len(nl), '| wavs', sum(x.endswith('.wav') for x in nl), '| empty transcripts', sum(1 for v in texts.values() if not v), '| MB', round(os.path.getsize(zp)/1e6))
print('>>> Download /workspace/work/submission_e13.zip and e13_D.pt, THEN stop/terminate the pod <<<')

In [ ]:
# 9) SECOND SHOT: ensemble E13 + D (TTA x2 each = 4 passes, ~75 min, ~$2.3). Run only if E13 was close to or better than D in cell 6.
import soundfile as sf, zipfile, time
from collections import Counter
from pathlib import Path
test_files = sorted(Path('/kaggle/input/datasets/avin55/indoml-track2-test/audio').glob('*.wav')); print('test clips', len(test_files))
sub = Path('/workspace/work/t2_submission_ens'); sub.mkdir(exist_ok=True); fails = Counter(); t0 = time.time(); parts = ['E', 'D']; MODELS.update({'E': mE, 'D': mD}); KW2 = dict(shifts=(0, 50), win=8.0, hop=6.0)
for k, p in enumerate(test_files):
    w = data.read_audio(p); y = post(w, np.mean([enhance_tta(MODELS[q], w, **KW2) for q in parts], 0)); c = check_output(w, y)
    for key in ['finite', 'len_match', 'mono', 'not_silent', 'level_ok', 'no_clip']:
        if not c[key]: fails[key] += 1
    assert c['finite'] and c['len_match'] and c['mono'], p.name
    sf.write(sub / p.name, y, 16000, subtype='PCM_16')
    if k % 500 == 0: print(k, f'{(time.time()-t0)/60:.1f} min', flush=True)
print('enhanced in', round((time.time()-t0)/60, 1), 'min | fails', dict(fails), '| combo', BEST)
names = [p.name for p in test_files]; texts = {}
for i in range(0, len(names), ASR_BS):
    for n, t in zip(names[i:i+ASR_BS], asr.transcribe_paths([str(sub / n) for n in names[i:i+ASR_BS]])): texts[n[:-4]] = (t or '').strip()
with open('/workspace/work/transcripts_ens.jsonl', 'w', encoding='utf-8') as f:
    for n in names: f.write(json.dumps({'clip_id': n[:-4], 'text': texts.get(n[:-4], '')}, ensure_ascii=False) + '\n')
zp = '/workspace/work/submission_e13_plus_D.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/workspace/work/transcripts_ens.jsonl', 'transcripts.jsonl')
    for n in names: z.write(sub / n, n)
nl = zipfile.ZipFile(zp).namelist(); print('ZIP entries', len(nl), '| wavs', sum(x.endswith('.wav') for x in nl), '| empty transcripts', sum(1 for v in texts.values() if not v), '| MB', round(os.path.getsize(zp)/1e6))
print('>>> Download /workspace/work/submission_e13_plus_D.zip and e12_D.pt, THEN stop/terminate the pod <<<')